In [ ]:
from typing import Any, Dict, List, Optional
from core.config import RAGConfig, DEFAULT_CONFIG
import logging
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.indexing import index, InMemoryRecordManager, IndexingResult
from langchain_core.retrievers import BaseRetriever
import torch
from langchain_text_splitters import MarkdownHeaderTextSplitter
import uuid
import hashlib
from pathlib import Path
import json
import sys
import chromadb

logger = logging.getLogger(__name__)

logger.setLevel(logging.INFO)

if not logger.hasHandlers():
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)


COLLECTION_NAME = "knowledge_base"


class VectorStoreBuilder:
    def __init__(
        self,
        model_name: str,
        index_save_path: str,
    ):
        self.model_name = model_name
        self.index_save_path = index_save_path
        self.embeddings = None
        self.vectorstore: Optional[Chroma] = None
        self._client: Optional[chromadb.ClientAPI] = None
        self._record_manager: Optional[InMemoryRecordManager] = None
        self.setup_embeddings()

    def setup_embeddings(self):
        """Setup embeddings with local file cache"""
        logger.info(f"Initializing embedding model: {self.model_name}")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.model_name,
            model_kwargs={"device": device},
            encode_kwargs={"normalize_embeddings": True},
        )
        logger.info(f"Embedding model initialized on {device}")

    def _get_record_manager(self) -> InMemoryRecordManager:
        if self._record_manager is None:
            self._record_manager = InMemoryRecordManager(
                namespace=COLLECTION_NAME,
            )
            self._record_manager.create_schema()
        return self._record_manager

    def _get_client(self) -> chromadb.ClientAPI:
        """
        Get the ChromaDB client.

        Returns:
            ChromaDB client
        """
        if not self._client:
            Path(self.index_save_path).mkdir(parents=True, exist_ok=True)
            self._client = chromadb.PersistentClient(path=self.index_save_path)
        return self._client

    def _get_vectorstore(self) -> Chroma:
        """
        Get the Chroma vector store.

        Returns:
            Chroma vector store
        """
        if not self.vectorstore:
            client = self._get_client()
            self.vectorstore = Chroma(
                client=client,
                collection_name=COLLECTION_NAME,
                embedding_function=self.embeddings,
            )
        return self.vectorstore

    def collection_exists(self) -> bool:
        """
        Check if the collection exists in the vector store.

        Returns:
            True if the collection exists, False otherwise
        """
        try:
            client = self._get_client()
            collection = client.get_collection(name=COLLECTION_NAME)
            return collection.count() > 0
        except Exception:
            return False

    def index_documents(
        self,
        chunks: List[Document],
        cleanup: Literal["incremental", "full", "scoped_full"] = "incremental",
    ) -> IndexingResult:
        """
        Index documents using langchain's index() API with automatic deduplication and change detection.

        Args:
            chunks: List of Document objects to be indexed
            cleanup: Cleanup mode - "incremental", "full", or "scoped_full"

        Returns:
            Indexing result
        """
        if not chunks:
            raise ValueError("Document chunk list cannot be empty")

        logger.info(f"Indexing {len(chunks)} documents with cleanup mode: {cleanup}...")

        vectorstore = self._get_vectorstore()
        record_manager = self._get_record_manager()

        result = index(
            docs_source=chunks,
            record_manager=record_manager,
            vector_store=vectorstore,
            cleanup=cleanup,
            source_id_key="source",
            key_encoder="sha256"
        )

        logger.info(
            f"Indexing complete: +{result['num_added']} ~{result['num_updated']} "
            f"-{result['num_deleted']} = {result['num_skipped']}"
        )
        return result

    def get_all_documents(self) -> List[Document]:
        """
        Retrieve all documents from the index

        Returns:
            List of all documents in the index
        """
        client = self._get_client()

        try:
            collection = client.get_collection(name=COLLECTION_NAME)
        except Exception as e:
            logger.error(f"Error retrieving documents: {e}")
            return []

        results = collection.get(include=["documents", "metadatas"])
        documents = []
        for doc_text, metadata in zip(results["documents"], results["metadatas"]):
            documents.append(Document(page_content=doc_text, metadata=metadata))

        logger.info(f"Retrieved {len(documents)} documents from the index")
        return documents

    def get_vectorstore(self) -> Chroma:
        """
        Get the vector store object

        Returns:
            Vector store object
        """
        return self._get_vectorstore()

    def reset_collection(self):
        """
        Reset the vector store collection
        """
        client = self._get_client()
        try:
            client.delete_collection(name=COLLECTION_NAME)
        except Exception as e:
            logger.error(f"Error resetting collection: {e}")
            pass
        self.vectorstore = None

    def similarity_search(self, query: str, k: int = 5) -> List[Document]:
        """
        Similarity search

        Args:
            query: Query text
            k: Number of results to return

        Returns:
            List of similar documents
        """
        vectorstore = self._get_vectorstore()

        return vectorstore.similarity_search(query, k=k)


In [3]:
class DocumentChunker:
    """
    Data preparation class - responsible for knowledge base data loading, cleaning, and preprocessing for RAG system.
    """

    def __init__(self, data_path: str) -> None:
        """
        Initialize data preparation class.

        Args:
            data_path: Root directory path of the knowledge base.
        """
        self.data_path = data_path
        self.documents: List[Document] = []
        self.chunks: List[Document] = []
        self.parent_child_map: Dict[str, str] = {}

    def load_documents(self, file_paths: Optional[List[str]] = None) -> List[Document]:
        """
        Load documents from the knwoledge base directory

        Args:
            file_paths: List of file paths to load. If None, all files in the data directory will be loaded.

        Returns:
            List of loaded documents
        """
        logger.info(f"Loading documents from {self.data_path}")

        documents = []
        data_path_obj = Path(self.data_path)

        if file_paths is not None:
            md_files = [data_path_obj / fp for fp in file_paths]
        else:
            md_files = list(data_path_obj.rglob("*.md"))

        for md_file in md_files:
            try:
                with open(file=md_file, mode="r", encoding="utf-8") as f:
                    content = f.read()

                try:
                    data_root = Path(self.data_path).resolve()
                    relative_path = (
                        Path(md_file).resolve().relative_to(data_root).as_posix()
                    )
                except Exception:
                    relative_path = Path(md_file).as_posix()

                parent_id = hashlib.md5(relative_path.encode("utf-8")).hexdigest()

                doc = Document(
                    page_content=content,
                    metadata={
                        "source": str(md_file),
                        "parent_id": parent_id,
                        "doc_type": "parent",
                    },
                )

                documents.append(doc)

            except Exception as e:
                logger.error(f"Error reading file {md_file}: {e}")

        for doc in documents:
            self._enhace_metadata(doc)

        self.documents = documents
        logger.info(f"Successfully loaded {len(documents)} documents")
        return documents

    def _enhace_metadata(self, doc: Document) -> None:
        """
        Enhance document metadata - extract category information from file path hierarchy

        Args:
            doc: Document to enhance metadata for
        """
        file_path = Path(doc.metadata.get("source", ""))
        data_root = Path(self.data_path).resolve()

        try:
            relative = file_path.resolve().relative_to(data_root)
            path_hierarchy = list(relative.parent.parts)
        except ValueError, RuntimeError:
            path_hierarchy = []

        doc.metadata["doc_name"] = file_path.stem
        doc.metadata["path_hierarchy"] = path_hierarchy
        doc.metadata["primary_category"] = (
            path_hierarchy[0] if len(path_hierarchy) >= 1 else ""
        )
        doc.metadata["subcategory"] = (
            path_hierarchy[1] if len(path_hierarchy) >= 2 else ""
        )

    def chunk_documents(self) -> List[Document]:
        """
        Markdown structure-aware chunking

        Returns:
            List of chunked documents
        """
        logger.info("Performing Markdown structure-aware chunking...")

        if not self.documents:
            raise ValueError("Please load documents first.")

        chunks = self._markdown_header_split()

        for i, chunk in enumerate(chunks):
            if "chunk_id" not in chunk.metadata:
                chunk.metadata["chunk_id"] = str(uuid.uuid4())
            chunk.metadata["batch_index"] = i
            chunk.metadata["chunk_size"] = len(chunk.page_content)

        self.chunks = chunks
        logger.info(f"Markdown chunking complete, generated {len(chunks)} chunks")
        return chunks

    def _markdown_header_split(self) -> List[Document]:
        """
        Perform structured splitting using Markdown header splitter

        Returns:
            List of documents split by header structure
        """
        headers_to_split_on = [
            ("#", "h1"),
            ("##", "h2"),
            ("###", "h3"),
        ]

        markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=headers_to_split_on,
            strip_headers=False,
        )

        all_chunks = []

        for doc in self.documents:
            try:
                content_preview = doc.page_content[:200]
                has_headers = any(
                    line.strip().startswith("#") for line in content_preview.split("\n")
                )

                if not has_headers:
                    logger.warning(
                        f"Document {doc.metadata.get('doc_name', 'unknown')} has no Markdown headers"
                    )
                    logger.debug(f"Content preview: {content_preview}")

                md_chunks = markdown_splitter.split_text(doc.page_content)

                logger.debug(
                    f"Document {doc.metadata.get('doc_name', 'unknown')} split into {len(md_chunks)} chunks"
                )

                if len(md_chunks) <= 1:
                    logger.warning(
                        f"Document {doc.metadata.get('doc_name', 'unknown')} could not split by headers, may lack header structure"
                    )

                parent_id = doc.metadata["parent_id"]

                for i, chunk in enumerate(md_chunks):
                    child_id = str(uuid.uuid4())
                    chunk.metadata.update(doc.metadata)
                    chunk.metadata.update(
                        {
                            "chunk_id": child_id,
                            "parent_id": parent_id,
                            "doc_type": "child",
                            "chunk_index": i,
                        }
                    )

                    self.parent_child_map[child_id] = parent_id

                all_chunks.extend(md_chunks)

            except Exception as e:
                logger.warning(
                    f"Markdown splitting failed for {doc.metadata.get('source', 'unknown')}: {e}"
                )
                all_chunks.append(doc)

        logger.info(
            f"Markdown structure splitting complete, generated {len(all_chunks)} structured chunks"
        )
        return all_chunks

    def filter_documents(self, field: str, value: str) -> List[Document]:
        """
        Filter documents by metadata field.

        Args:
            field: Metadata field name (e.g. primary_category, subcategory, doc_name)
            value: Expected field Value

        Returns:
            Filtered list of documents
        """
        return [doc for doc in self.documents if doc.metadata.get(field) == value]

    def get_statistics(self) -> Dict[str, Any]:
        """
        Get data statistics

        Returns:
            Dictionary of statistics
        """
        if not self.documents:
            return {}

        primary_categories: Dict[str, int] = {}
        subcategories: Dict[str, int] = {}

        for doc in self.documents:
            pc = doc.metadata.get("primary_category", "")
            sc = doc.metadata.get("subcategory", "")
            primary_categories[pc] = primary_categories.get(pc, 0) + 1
            if sc:
                subcategories[sc] = subcategories.get(sc, 0) + 1

        return {
            "total_documents": len(self.documents),
            "total_chunks": len(self.chunks),
            "primary_categories": primary_categories,
            "subcategories": subcategories,
            "avg_chunk_size": sum(
                chunk.metadata.get("chunk_size", 0) for chunk in self.chunks
            )
            / len(self.chunks)
            if self.chunks
            else 0,
        }

    @staticmethod
    def get_statistics_from_chunks(chunks: List[Document]) -> Dict[str, Any]:
        """
        Get statistics from a list of document chunks.

        Args:
            chunks: List of document chunks

        Returns:
            Dictionary of statistics
        """
        if not chunks:
            return {}

        parent_ids = set()
        primary_categories: Dict[str, int] = {}
        subcategories: Dict[str, int] = {}

        for chunk in chunks:
            parent_id = chunk.metadata.get("parent_id", "")
            if parent_id:
                parent_ids.add(parent_id)

            pc = chunk.metadata.get("primary_category", "")
            sc = chunk.metadata.get("subcategory", "")
            if pc:
                primary_categories[pc] = primary_categories.get(pc, 0) + 1
            if sc:
                subcategories[sc] = subcategories.get(sc, 0) + 1

        return {
            "total_documents": len(parent_ids),
            "total_chunks": len(chunks),
            "primary_categories": primary_categories,
            "subcategories": subcategories,
            "avg_chunk_size": sum(
                chunk.metadata.get("chunk_size", 0) for chunk in chunks
            )
            / len(chunks)
            if chunks
            else 0,
        }

    def export_metadata(self, output_path: str):
        """
        Export metadata to a JSON file.

        Args:
            output_path: Output file path
        """

        metadata_list = []
        for doc in self.documents:
            metadata_list.append(
                {
                    "source": doc.metadata.get("source"),
                    "doc_name": doc.metadata.get("doc_name"),
                    "primary_category": doc.metadata.get("primary_category"),
                    "subcategory": doc.metadata.get("subcategory"),
                    "path_hierarchy": doc.metadata.get("path_hierarchy"),
                    "content_length": len(doc.page_content),
                }
            )

        with open(file=output_path, mode="w", encoding="utf-8") as f:
            json.dump(metadata_list, f, ensure_ascii=False, indent=2)

        logger.info(f"Metadata exported to: {output_path}")

    def get_parent_documents(self, child_chunks: List[Document]) -> List[Document]:
        """
        Retrieves parent documents from child chunks (with smart deduplication).

        Args:
            child_chunks: List of retrieved child chunks

        Returns:
            Deduplicated list of parent documents, sorted by relevance
        """
        parent_relevance = {}
        parent_docs_map = {}

        for chunk in child_chunks:
            parent_id = chunk.metadata.get("parent_id")
            if parent_id:
                parent_relevance[parent_id] = parent_relevance.get(parent_id, 0) + 1

                if parent_id not in parent_docs_map:
                    for doc in self.documents:
                        if doc.metadata.get("parent_id") == parent_id:
                            parent_docs_map[parent_id] = doc
                            break

        sorted_parent_ids = sorted(
            parent_relevance.keys(), key=lambda x: parent_relevance[x], reverse=True
        )

        parent_docs = []
        for parent_id in sorted_parent_ids:
            if parent_id in parent_docs_map:
                parent_docs.append(parent_docs_map[parent_id])

        parent_info = []
        for doc in parent_docs:
            doc_name = doc.metadata.get("doc_name", "unknown")
            parent_id = doc.metadata.get("parent_id")
            relevance_count = parent_relevance.get(parent_id, 0)
            parent_info.append(f"{doc_name}({relevance_count}块)")

        logger.info(
            f"Found {len(child_chunks)} deduplicated parent documents from {len(parent_docs)} child chunks: {', '.join(parent_info)}"
        )
        return parent_docs


In [5]:
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from rank_bm25 import BM25Okapi
from pydantic import Field, PrivateAttr

class HybridRetriever(BaseRetriever):
    vectorstore: Chroma
    chunks: List[Document] = Field(default_factory=list)
    vector_k: int = 5
    bm25_k: int = 5
    top_k: int = 3
    rrf_constant: int = 60

    _bm25_index: Optional[BM25Okapi] = PrivateAttr(default=None)
    _tokenized_corpus: Optional[List[List[str]]] = PrivateAttr(default=None)

    def _get_bm25_index(self) -> BM25Okapi:
        if self._bm25_index is None:
            logger.info("Building BM25 index (first use)...")
            self._tokenized_corpus = [doc.page_content.split() for doc in self.chunks]
            self._bm25_index = BM25Okapi(self._tokenized_corpus)
            logger.info("BM25 index built successfully")
        return self._bm25_index

    def _bm25_search(self, query: str) -> List[Document]:
        bm25 = self._get_bm25_index()
        tokenized_query = query.split()
        scores = bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[
            : self.bm25_k
        ]
        return [self.chunks[i] for i in top_indices if scores[i] > 0]

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        vector_docs = self.vectorstore.similarity_search(query, k=self.vector_k)
        bm25_docs = self._bm25_search(query)
        return self._rrf_rerank(vector_docs, bm25_docs)[: self.top_k]

    def _rrf_rerank(
        self, vector_docs: List[Document], bm25_docs: List[Document]
    ) -> List[Document]:
        k = self.rrf_constant
        doc_scores: Dict[str, float] = {}
        doc_objects: Dict[str, Document] = {}

        for rank, doc in enumerate(vector_docs):
            doc_id = doc.metadata.get("chunk_id", str(hash(doc.page_content)))
            doc_objects[doc_id] = doc
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1.0 / (k + rank + 1)

        for rank, doc in enumerate(bm25_docs):
            doc_id = doc.metadata.get("chunk_id", str(hash(doc.page_content)))
            doc_objects[doc_id] = doc
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1.0 / (k + rank + 1)

        # Sort documents by their combined scores
        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        reranked = []
        for doc_id, score in sorted_docs:
            doc = doc_objects[doc_id]
            doc.metadata["rrf_score"] = score
            reranked.append(doc)

        logger.info(
            f"RRF reranking: {len(vector_docs)} vector + {len(bm25_docs)} BM25 -> {len(reranked)} merged"
        )
        return reranked


class RetrievalOptimization:
    """Retrieval Optimization"""

    def __init__(self, vectorstore: Chroma, chunks: List[Document]):
        """
        Initialize the RetrievalOptimization class.

        Args:
            vectorstore: Chroma vector store
            chunks: List of documents
        """
        self.vectorstore = vectorstore
        self.chunks = chunks
        self.hybrid_retriever = HybridRetriever(
            vectorstore=self.vectorstore, chunks=self.chunks
        )

    def hybrid_search(self, query: str, top_k: int = 3) -> List[Document]:
        """
        Hybrid search - combine vector search and BM25 search, use RRF for reranking

        Args:
            query: Query text
            top_k: Number of results to return

        Returns:
            List of retrieved documents
        """

        self.hybrid_retriever.top_k = top_k
        return self.hybrid_retriever.invoke(query)

    def metadata_filtered_search(
        self, query: str, filters: Dict[str, Any], top_k: int = 5
    ) -> List[Document]:
        """
        Metadata-filtered search

        Args:
            query: Query text
            filters: Metadata filter conditions
            top_k: Number of results to return

        Returns:
            Filtered list of documents
        """

        try:
            return self.vectorstore.similarity_search(query, k=top_k, filters=filters)
        except Exception as e:
            logger.error(f"Native filter failed ({e}), falling back to post-filtering")
            docs = self.hybrid_search(query, top_k * 3)
            filtered = [
                doc
                for doc in docs
                if all(doc.metadata.get(k) == v for k, v in filters.items())
            ]
            return filtered[:top_k]


In [6]:

class RAGSystem:
    def __init__(self, config: Optional[RAGConfig] = None):
        """
        Initialize the RAG system.

        Args:
            config: Configuration for the RAG system.
        """
        logger.info("🚀 Initializing RAG system...")

        self.config: RAGConfig = config or DEFAULT_CONFIG
        self.chunks: List[Document] = []
        self.retrieval: Optional[RetrievalOptimization] = None

        self.doc_chunker = DocumentChunker(self.config.knowledge_base_path)
        self.vector_builder = VectorStoreBuilder(
            model_name=self.config.embedding_model,
            index_save_path=self.config.index_save_path,
        )

    def build(self) -> IndexingResult:
        logger.info("Loading and chunking docuements")
        self.doc_chunker.load_documents()
        self.chunks = self.doc_chunker.chunk_documents()

        logger.info(f"Indexing {len(self.chunks)} chunks...")
        result = self.vector_builder.index_documents(self.chunks)

        vectorstore = self.vector_builder.get_vectorstore()
        self.retrieval = RetrievalOptimization(
            vectorstore=vectorstore, chunks=self.chunks
        )

        logger.info("Knowledge base ready")
        return result

    def search(self, query: str, top_k: Optional[int] = None) -> List[Document]:
        """
        Search the knowledge base with a query.

        Args:
            query: Query text
            top_k: Number of results to return

        Returns:
            List of retrieved documents
        """
        if not self.retrieval:
            raise RuntimeError("Please call build() first")

        k = top_k or self.config.top_k
        return self.retrieval.hybrid_search(query, top_k=k)


# query = "FAISS是做什么的？"
query = "How to perform inner join with SQLAlchemy in Python?"
#query = "天然珠光云母水解过程PH值？"
rag = RAGSystem()
rag.build()

results = rag.search(query, top_k=5)

logger.info(f"User query: '{query}'")
logger.info("Similarity search results:")
for i, doc in enumerate(results):
    logger.info(f"{i + 1}. {doc.page_content}")


2026-05-31 14:09:40,958 - INFO - 🚀 Initializing RAG system...
2026-05-31 14:09:40,959 - INFO - Initializing embedding model: D:/modelscope/hub/models/microsoft/harrier-oss-v1-0___6b


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2806.72it/s]


2026-05-31 14:09:46,822 - INFO - Embedding model initialized on cuda
2026-05-31 14:09:46,823 - INFO - Loading and chunking docuements
2026-05-31 14:09:46,823 - INFO - Loading documents from ./knowledge_base
2026-05-31 14:09:46,835 - INFO - Successfully loaded 8 documents
2026-05-31 14:09:46,835 - INFO - Performing Markdown structure-aware chunking...
2026-05-31 14:09:46,839 - INFO - Markdown structure splitting complete, generated 92 structured chunks
2026-05-31 14:09:46,840 - INFO - Markdown chunking complete, generated 92 chunks
2026-05-31 14:09:46,841 - INFO - Indexing 92 chunks...
2026-05-31 14:09:46,841 - INFO - Indexing 92 documents with cleanup mode: incremental...
2026-05-31 14:09:55,911 - INFO - Indexing complete: +92 ~0 -0 = 0
2026-05-31 14:09:55,912 - INFO - Knowledge base ready
2026-05-31 14:09:56,117 - INFO - Building BM25 index (first use)...
2026-05-31 14:09:56,120 - INFO - BM25 index built successfully
2026-05-31 14:09:56,121 - INFO - RRF reranking: 5 vector + 5 BM25 ->